In [ ]:
"""
DSO 530 Group Project 2026
Exploratory Data Analysis — Visualization Script
=================================================
Produces a single 4×3 figure (eda_overview.png) covering:
  1.  CS class balance (pie)
  2.  LC distribution — log scale, non-zero only
  3.  Claim rate by vehicle type (X.19)
  4.  Claim rate by loyalty years (X.8)
  5.  Claim rate by # policies held (X.9)
  6.  Claim rate by insured age
  7.  Claim rate — fuel / payment / driver comparisons
  8.  Net premium distribution by CS (X.14)
  9.  Claim rate by vehicle value (X.25)
  10. Correlation heatmap (numeric features + targets)

Run standalone:
    python eda_visualization.py
Or import the helper functions:
    from eda_visualization import prepare_eda_data, plot_eda
"""

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
TRAIN_PATH = 'data/raw/insurance_train2026.csv'
OUTPUT_PATH = 'figures/eda_overview.png'
REF_DATE    = pd.Timestamp('2019-12-31')

# ── colour palette (mirrors data_processing.py style) ──────────────
C_CLAIM = '#E8593C'   # coral-red  — high risk / has claim
C_NO    = '#1D9E75'   # teal-green — low risk  / no claim
C_BAR   = '#378ADD'   # blue       — single-series bars
C_HIST  = '#7F77DD'   # purple     — histogram fill
C_GRID  = '#E8E7E3'
C_TEXT  = '#2C2C2A'
C_MUTED = '#5F5E5A'

AVG_THRESHOLD = 0.112   # overall claim rate ≈ 11.2%

# ─────────────────────────────────────────────
# 1. DATA PREPARATION
# ─────────────────────────────────────────────
def prepare_eda_data(path: str = TRAIN_PATH) -> pd.DataFrame:
    """
    Load raw training CSV and add derived columns needed for EDA.
    Does NOT alter the pipeline-processed feature matrix — this is
    purely for exploratory summaries and plots.
    """
    df = pd.read_csv(path)

    # ── target variables ──────────────────────────────────────────
    has_claim   = df['X.16'] > 0
    df['CS']    = has_claim.astype(int)
    df['LC']    = np.where(has_claim, df['X.15'] / df['X.16'], 0.0)
    df['HALC']  = np.where(has_claim, df['LC'] * df['X.18'],   0.0)

    # ── date-derived columns ──────────────────────────────────────
    df['insured_age'] = (
        REF_DATE - pd.to_datetime(df['X.5'], dayfirst=True)
    ).dt.days / 365.25

    df['license_yrs'] = (
        REF_DATE - pd.to_datetime(df['X.6'], dayfirst=True)
    ).dt.days / 365.25

    # ── simple derived ────────────────────────────────────────────
    df['vehicle_age'] = 2019 - df['X.22']

    # ── binned variables for grouped charts ───────────────────────
    df['loyalty_bucket'] = pd.cut(
        df['X.8'], bins=[0, 1, 2, 5, 10, 40],
        labels=['1yr', '2yr', '3–5yr', '6–10yr', '10yr+'])

    df['n_pol_bucket'] = pd.cut(
        df['X.9'], bins=[0, 1, 2, 3, 5, 20],
        labels=['1', '2', '3', '4–5', '6+'])

    df['age_bin'] = pd.cut(
        df['insured_age'], bins=[0, 30, 40, 50, 60, 70, 110],
        labels=['<30', '30s', '40s', '50s', '60s', '70+'])

    df['value_bucket'] = pd.cut(
        df['X.25'],
        bins=[0, 8000, 12000, 16000, 22000, 30000, 250000],
        labels=['<8k', '8–12k', '12–16k', '16–22k', '22–30k', '>30k'])

    return df


# ─────────────────────────────────────────────
# 2. PLOT HELPERS
# ─────────────────────────────────────────────
def _apply_base_style():
    plt.rcParams.update({
        'axes.facecolor':      'white',
        'figure.facecolor':    'white',
        'axes.spines.top':     False,
        'axes.spines.right':   False,
        'axes.spines.left':    False,
        'axes.spines.bottom':  True,
        'axes.edgecolor':      '#B4B2A9',
        'axes.grid':           True,
        'grid.color':          C_GRID,
        'grid.linewidth':      0.6,
        'xtick.color':         C_MUTED,
        'ytick.color':         C_MUTED,
        'text.color':          C_TEXT,
    })


def _color_by_threshold(values, threshold=AVG_THRESHOLD):
    """Return coral for above-average claim rate, teal for below."""
    return [C_CLAIM if v > threshold else C_NO for v in values]


def _label_bars(ax, bars, fmt='{:.1%}', offset=0.002, horizontal=False):
    """Annotate bar chart with value labels."""
    for bar in bars:
        if horizontal:
            w = bar.get_width()
            ax.text(w + offset, bar.get_y() + bar.get_height() / 2,
                    fmt.format(w), va='center', ha='left',
                    fontsize=9, color=C_MUTED)
        else:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + offset,
                    fmt.format(h), ha='center', va='bottom',
                    fontsize=9, color=C_MUTED)


def _add_avg_line(ax, value, horizontal=False):
    if horizontal:
        ax.axvline(value, color=C_MUTED, lw=1.2, linestyle='--',
                   label=f'Overall {value:.1%}')
    else:
        ax.axhline(value, color=C_MUTED, lw=1.2, linestyle='--',
                   label=f'Overall {value:.1%}')


# ─────────────────────────────────────────────
# 3. INDIVIDUAL PANEL FUNCTIONS
# ─────────────────────────────────────────────
def _panel_pie(ax, df):
    """Panel 1 — CS class balance pie chart."""
    counts = df['CS'].value_counts().sort_index()
    wedges, _, autotexts = ax.pie(
        counts,
        labels=['No Claim\n(CS=0)', 'Has Claim\n(CS=1)'],
        colors=[C_NO, C_CLAIM],
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
        textprops={'fontsize': 10},
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_fontweight('bold')
        at.set_color('white')
    ax.set_title('Claim status distribution\n(CS)', fontsize=11,
                 fontweight='bold', pad=8)


def _panel_lc_hist(ax, df):
    """Panel 2 — LC log-scale histogram (non-zero only)."""
    nz = df.loc[df['LC'] > 0, 'LC']
    ax.hist(np.log1p(nz), bins=50, color=C_HIST, alpha=0.85,
            edgecolor='white', linewidth=0.3)
    ax.axvline(np.log1p(nz.median()), color=C_CLAIM, lw=1.5,
               linestyle='--', label=f'Median  {nz.median():.0f}')
    ax.set_title('LC distribution (log scale)\nnon-zero claims only',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('log(1 + LC)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.tick_params(bottom=False)
    ax.legend(fontsize=9)


def _panel_risk_type(ax, df):
    """Panel 3 — Claim rate by vehicle type (X.19)."""
    labels = {1: 'Motorbike', 2: 'Van', 3: 'Car', 4: 'Agricultural'}
    grp = df.groupby('X.19')['CS'].mean()
    colors = _color_by_threshold(grp.values)
    bars = ax.bar([labels[i] for i in grp.index], grp.values,
                  color=colors, edgecolor='white', linewidth=0.5, width=0.6)
    _add_avg_line(ax, df['CS'].mean())
    ax.set_title('Claim rate by vehicle type\n(X.19)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Claim rate', fontsize=9)
    ax.set_ylim(0, 0.22)
    ax.tick_params(bottom=False)
    _label_bars(ax, bars, offset=0.003)
    ax.legend(fontsize=8)


def _panel_loyalty(ax, df):
    """Panel 4 — Claim rate by loyalty years (X.8)."""
    grp = df.groupby('loyalty_bucket', observed=True)['CS'].mean()
    colors = _color_by_threshold(grp.values)
    bars = ax.bar(grp.index, grp.values, color=colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    _add_avg_line(ax, df['CS'].mean())
    ax.set_title('Claim rate by loyalty\n(X.8 — years with insurer)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Claim rate', fontsize=9)
    ax.set_ylim(0, 0.26)
    ax.tick_params(bottom=False)
    _label_bars(ax, bars, offset=0.003)


def _panel_n_policies(ax, df):
    """Panel 5 — Claim rate by number of policies held (X.9)."""
    grp = df.groupby('n_pol_bucket', observed=True)['CS'].mean()
    colors = _color_by_threshold(grp.values)
    bars = ax.bar(grp.index, grp.values, color=colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    _add_avg_line(ax, df['CS'].mean())
    ax.set_title('Claim rate by # policies held\n(X.9)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Claim rate', fontsize=9)
    ax.set_ylim(0, 0.42)
    ax.tick_params(bottom=False)
    _label_bars(ax, bars, offset=0.004)


def _panel_age(ax, df):
    """Panel 6 — Claim rate by insured age bin."""
    grp = df.groupby('age_bin', observed=True)['CS'].mean()
    colors = _color_by_threshold(grp.values)
    bars = ax.bar(grp.index, grp.values, color=colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    _add_avg_line(ax, df['CS'].mean())
    ax.set_title('Claim rate by insured age\n(X.5 → derived)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Claim rate', fontsize=9)
    ax.set_ylim(0, 0.20)
    ax.tick_params(bottom=False)
    _label_bars(ax, bars, offset=0.002)


def _panel_categorical_compare(ax, df):
    """Panel 7 — Horizontal bar: fuel / payment / driver comparisons."""
    avg = df['CS'].mean()
    cats = {
        'Petrol (P)':    df[df['X.27'] == 'P']['CS'].mean(),
        'Diesel (D)':    df[df['X.27'] == 'D']['CS'].mean(),
        'Annual pay':    df[df['X.13'] == 0]['CS'].mean(),
        'Half-yearly':   df[df['X.13'] == 1]['CS'].mean(),
        'Single driver': df[df['X.21'] == 0]['CS'].mean(),
        'Multi driver':  df[df['X.21'] == 1]['CS'].mean(),
    }
    colors = _color_by_threshold(cats.values())
    bars = ax.barh(list(cats.keys()), list(cats.values()),
                   color=colors, edgecolor='white', linewidth=0.5, height=0.6)
    _add_avg_line(ax, avg, horizontal=True)
    ax.set_title('Claim rate — categorical\ncomparisons',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Claim rate', fontsize=9)
    ax.set_xlim(0, 0.22)
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False)
    _label_bars(ax, bars, fmt='{:.1%}', offset=0.003, horizontal=True)
    ax.legend(fontsize=8)


def _panel_premium_hist(ax, df):
    """Panel 8 — Net premium distribution by claim status (X.14)."""
    bins = np.linspace(0, 900, 45)
    for cs_val, color, label in [(0, C_NO, 'No claim'), (1, C_CLAIM, 'Has claim')]:
        ax.hist(df.loc[df['CS'] == cs_val, 'X.14'].clip(upper=900),
                bins=bins, alpha=0.65, color=color, label=label,
                density=True, edgecolor='white', linewidth=0.2)
    ax.set_title('Net premium distribution\nby claim status (X.14)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Net premium (clipped at 900)', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.tick_params(bottom=False)
    ax.legend(fontsize=9)


def _panel_vehicle_value(ax, df):
    """Panel 9 — Claim rate by vehicle market value (X.25)."""
    grp = df.groupby('value_bucket', observed=True)['CS'].mean()
    colors = _color_by_threshold(grp.values)
    bars = ax.bar(grp.index, grp.values, color=colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    _add_avg_line(ax, df['CS'].mean())
    ax.set_title('Claim rate by vehicle value\n(X.25)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Claim rate', fontsize=9)
    ax.set_ylim(0, 0.19)
    ax.tick_params(axis='x', labelsize=8, bottom=False)
    _label_bars(ax, bars, offset=0.002)


def _panel_corr_heatmap(ax, df):
    """Panel 10 — Correlation heatmap of numeric features + targets."""
    num_cols = [
        'X.8',  'X.9',  'X.10', 'X.11', 'X.12',
        'X.14', 'X.23', 'X.24', 'X.25', 'X.28',
        'vehicle_age', 'insured_age', 'license_yrs', 'CS', 'LC',
    ]
    col_labels = [
        'Loyalty yrs', '# Policies', 'Max policies', 'Max products', 'Cancellations',
        'Premium',     'Horsepower', 'Cylinder cc',  'Vehicle value', 'Weight',
        'Vehicle age', 'Insured age', 'License yrs',  'CS (target)',  'LC (target)',
    ]
    corr = df[num_cols].corr()

    cmap = mcolors.LinearSegmentedColormap.from_list(
        'teal_coral', ['#E8593C', '#F5F4F0', '#1D9E75'], N=256)
    im = ax.imshow(corr.values, cmap=cmap, vmin=-0.6, vmax=0.6, aspect='auto')

    ax.set_xticks(range(len(col_labels)))
    ax.set_yticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=40, ha='right', fontsize=8.5)
    ax.set_yticklabels(col_labels, fontsize=8.5)
    ax.spines[:].set_visible(False)
    ax.tick_params(bottom=False, left=False)
    ax.set_title('Correlation matrix — numeric features + targets',
                 fontsize=11, fontweight='bold', pad=10)

    for i in range(len(col_labels)):
        for j in range(len(col_labels)):
            v = corr.values[i, j]
            if abs(v) > 0.08:
                txt_color = 'white' if abs(v) > 0.35 else C_TEXT
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=7.5, color=txt_color,
                        fontweight='bold' if abs(v) > 0.3 else 'normal')

    plt.colorbar(im, ax=ax, shrink=0.6, pad=0.01, label='Pearson r')


# ─────────────────────────────────────────────
# 4. MASTER PLOT FUNCTION
# ─────────────────────────────────────────────
def plot_eda(df: pd.DataFrame, output_path: str = OUTPUT_PATH) -> None:
    """
    Compose all 10 panels into a single figure and save to disk.

    Parameters
    ----------
    df          : DataFrame returned by prepare_eda_data()
    output_path : Path for the output PNG file
    """
    _apply_base_style()

    fig = plt.figure(figsize=(18, 20), facecolor='white')
    fig.suptitle('Insurance Loss Analytics — EDA Overview',
                 fontsize=18, fontweight='bold', color=C_TEXT, y=0.98)

    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.38)

    # ── row 0 ──────────────────────────────────
    _panel_pie(            fig.add_subplot(gs[0, 0]), df)
    _panel_lc_hist(        fig.add_subplot(gs[0, 1]), df)
    _panel_risk_type(      fig.add_subplot(gs[0, 2]), df)

    # ── row 1 ──────────────────────────────────
    _panel_loyalty(        fig.add_subplot(gs[1, 0]), df)
    _panel_n_policies(     fig.add_subplot(gs[1, 1]), df)
    _panel_age(            fig.add_subplot(gs[1, 2]), df)

    # ── row 2 ──────────────────────────────────
    _panel_categorical_compare(fig.add_subplot(gs[2, 0]), df)
    _panel_premium_hist(       fig.add_subplot(gs[2, 1]), df)
    _panel_vehicle_value(      fig.add_subplot(gs[2, 2]), df)

    # ── row 3 (full width) ─────────────────────
    _panel_corr_heatmap(   fig.add_subplot(gs[3, :]),  df)

    # ── shared legend ──────────────────────────
    high = mpatches.Patch(color=C_CLAIM, label='Above avg claim rate')
    low  = mpatches.Patch(color=C_NO,   label='Below avg claim rate')
    fig.legend(handles=[high, low], loc='upper right', fontsize=9,
               framealpha=0.9, edgecolor='#D3D1C7',
               bbox_to_anchor=(0.99, 0.97))

    fig.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"[EDA] Figure saved → {output_path}")
    plt.close(fig)


# ─────────────────────────────────────────────
# 5. OPTIONAL: PRINT SUMMARY TABLES
# ─────────────────────────────────────────────
def print_eda_summary(df: pd.DataFrame) -> None:
    """Print the key tabular summaries to stdout."""
    avg = df['CS'].mean()
    print(f"\n{'='*55}")
    print(f"  CLAIM RATE SUMMARY  (overall avg = {avg:.1%})")
    print(f"{'='*55}")

    tables = {
        'Vehicle type (X.19)':      df.groupby('X.19')['CS'].agg(['mean','count']),
        'Loyalty bucket (X.8)':     df.groupby('loyalty_bucket', observed=True)['CS'].agg(['mean','count']),
        'Policies held (X.9)':      df.groupby('n_pol_bucket',   observed=True)['CS'].agg(['mean','count']),
        'Insured age':               df.groupby('age_bin',        observed=True)['CS'].agg(['mean','count']),
        'Vehicle value (X.25)':     df.groupby('value_bucket',   observed=True)['CS'].agg(['mean','count']),
        'Fuel type (X.27)':         df.groupby('X.27')['CS'].agg(['mean','count']),
        'Payment method (X.13)':    df.groupby('X.13')['CS'].agg(['mean','count']),
        'Multi-driver (X.21)':      df.groupby('X.21')['CS'].agg(['mean','count']),
    }
    for title, tbl in tables.items():
        print(f"\n{title}")
        print(tbl.rename(columns={'mean': 'claim_rate', 'count': 'n'}).round(3).to_string())

    print(f"\n{'─'*55}")
    print("Correlations with CS and LC (top numeric features):")
    num_cols = ['X.8','X.9','X.10','X.11','X.12','X.14',
                'X.23','X.24','X.25','X.28','vehicle_age','insured_age']
    corr_cs = df[num_cols + ['CS']].corr()['CS'].drop('CS')
    corr_lc = df[num_cols + ['LC']].corr()['LC'].drop('LC')
    summary = pd.DataFrame({'corr_CS': corr_cs, 'corr_LC': corr_lc})
    print(summary.sort_values('corr_CS', key=abs, ascending=False).round(3).to_string())


# ─────────────────────────────────────────────
# 6. ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == '__main__':
    print("[EDA] Loading data ...")
    df = prepare_eda_data(TRAIN_PATH)

    print("[EDA] Printing summary tables ...")
    print_eda_summary(df)

    print("\n[EDA] Generating figure ...")
    plot_eda(df, OUTPUT_PATH)

In [ ]:
from IPython.display import Image
Image('figures/eda_overview.png')